# How to Find Better Flight Options During Public Holidays for Students and Teachers

## 1. Project Overview

This project analyzes flight options for students and teachers who can only travel during major public holidays. It focuses on flights departing from Shanghai to Singapore, Seoul, and Tokyo during the Labour Day Holiday and National Day Holiday.

The aim is to identify flight options that better balance **price** and **comfort**, using indicators such as direct flight, early-morning departure, and number of stops.

## 2. Problem Definition

Students and teachers often have limited flexibility because they can only travel during public holidays. However, flights during these periods are often more expensive and less convenient.

This project investigates how flight prices and comfort-related features change before, during, and after public holidays, and how these patterns may help users make better travel decisions.

## 3. Target Users and Research Question

### Target Users
- Students
- Teachers
- Other travelers with limited flexibility during public holidays

### Main Research Question
How do flight prices and comfort-related features change around major public holidays, and how can students and teachers use this information to find better flight options?

### Sub-questions
1. How do prices change before, during, and after the holiday period?
2. Which destination tends to be more expensive or more affordable?
3. Are direct flights significantly more expensive?
4. How common are early-morning departures during holiday travel periods?
5. Which routes offer a better balance between price and comfort?

## 4. Data Source and Scope

### Data Scope
- Origin: Shanghai
- Destinations: Singapore, Seoul, Tokyo
- Holidays: Labour Day Holiday and National Day Holiday
- Time window: 7 days before, during, and 7 days after each holiday

### Comfort Indicators
In this project, comfort is measured using:
- whether the flight is direct
- whether the departure time is between 00:00 and 06:00
- number of stops

### Data Source
The dataset was collected from Skyscanner on 21 April 2026, containing flight search results for various departure dates around Labour Day and National Day holidays.

### Access Date
21 April 2026

## 5. Python Libraries

The analysis uses Python for data loading, cleaning, transformation, comparison, and visualization.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# Set style for better-looking plots
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

## 6. Load the Data

In this section, the dataset is loaded and inspected to understand its structure, variables, and possible data quality issues.

In [ ]:
# Read the Excel file
df = pd.read_excel('flight_data.xlsx', sheet_name='flight_data')

# Display basic info
print("First 5 rows:")
display(df.head())

print("\nDataset info:")
df.info()

print("\nMissing values:")
print(df.isnull().sum())

print(f"\nTotal rows: {len(df)}")

## 7. Initial Data Inspection

Before cleaning the dataset, it is important to check:
- whether the column names are clear
- whether there are missing values
- whether price, date, and time variables need conversion
- whether any variables need to be standardized

In [ ]:
# Make a copy to avoid modifying original
df_clean = df.copy()

# 1. Standardize column names (convert to lowercase and replace spaces with underscores)
df_clean.columns = df_clean.columns.str.lower().str.replace(' ', '_')

# 2. Keep only relevant columns for analysis
relevant_cols = ['origin_city', 'destination_city', 'departure_date', 'departure_time',
                 'price_cny', 'is_direct', 'num_stops', 'holiday', 'holiday_stage',
                 'is_red_eye', 'airline', 'arrival_time']
df_clean = df_clean[relevant_cols]

# 3. Clean price column (already numeric, but ensure it's int)
df_clean['price_cny'] = pd.to_numeric(df_clean['price_cny'], errors='coerce')

# 4. Convert departure_date to datetime
df_clean['departure_date'] = pd.to_datetime(df_clean['departure_date'])

# 5. Convert departure_time to datetime (to extract hour)
# First, ensure time is string, then parse
df_clean['departure_time_str'] = df_clean['departure_time'].astype(str)
# Extract hour as integer
df_clean['departure_hour'] = pd.to_datetime(df_clean['departure_time_str'], format='%H:%M:%S', errors='coerce').dt.hour

# 6. Handle missing values - drop rows with critical missing price or departure_hour
initial_rows = len(df_clean)
df_clean = df_clean.dropna(subset=['price_cny', 'departure_hour'])
print(f"Dropped {initial_rows - len(df_clean)} rows due to missing price or departure time.")

# 7. Check for duplicates and drop if any
df_clean = df_clean.drop_duplicates()
print(f"Rows after dropping duplicates: {len(df_clean)}")

# Display cleaned data info
print("\nCleaned dataset info:")
df_clean.info()
print("\nFirst 3 rows after cleaning:")
display(df_clean.head(3))

## 8. Feature Engineering

To support the analysis, several new variables are created:

- **holiday_stage**: before, during, or after the holiday (already present, but verified)
- **is_red_eye_numeric**: 1 if departure between 00:00-06:00, else 0
- **is_direct_numeric**: 1 if direct flight, else 0
- **num_stops**: already numeric

In [ ]:
# 1. Convert is_red_eye (yes/no) to numeric
df_clean['is_red_eye_numeric'] = df_clean['is_red_eye'].map({'yes': 1, 'no': 0})

# 2. Convert is_direct (yes/no) to numeric
df_clean['is_direct_numeric'] = df_clean['is_direct'].map({'yes': 1, 'no': 0})

# 3. Ensure num_stops is integer (it already is, but just in case)
df_clean['num_stops'] = df_clean['num_stops'].astype(int)

# 4. Create a simplified comfort score (higher = more comfortable)
# Simple rule: direct flight (+1), not red-eye (+1), and 0 stops (+1)
df_clean['comfort_score'] = (
    df_clean['is_direct_numeric'] +
    (1 - df_clean['is_red_eye_numeric']) +
    (df_clean['num_stops'] == 0).astype(int)   # 1 if num_stops == 0 else 0
)

# Check the new variables
print("Holiday stage distribution:")
print(df_clean['holiday_stage'].value_counts())
print("\nRed-eye distribution:")
print(df_clean['is_red_eye_numeric'].value_counts())
print("\nDirect flight distribution:")
print(df_clean['is_direct_numeric'].value_counts())
print("\nNum stops distribution:")
print(df_clean['num_stops'].value_counts())
print("\nComfort score distribution (0-3):")
print(df_clean['comfort_score'].value_counts().sort_index())

## 9. Exploratory Data Analysis

This section provides an overview of the dataset, including the distribution of destinations, prices, departure times, and comfort-related features.

In [ ]:
# 1. Destination distribution
print("Destination counts:")
print(df_clean['destination_city'].value_counts())

# 2. Holiday distribution
print("\nHoliday distribution:")
print(df_clean['holiday'].value_counts())

# 3. Price summary statistics
print("\nPrice summary (CNY):")
print(df_clean['price_cny'].describe())

# 4. Direct vs non-direct counts
print("\nDirect flight counts:")
print(df_clean['is_direct'].value_counts())

# 5. Red-eye vs non-red-eye counts
print("\nRed-eye flight counts:")
print(df_clean['is_red_eye'].value_counts())

# 6. Number of stops distribution
print("\nNumber of stops distribution:")
print(df_clean['num_stops'].value_counts().sort_index())

## 10. Price Analysis

This section compares ticket prices across:
- different destinations
- different holiday periods
- different stages of the holiday window (before, during, after)

In [ ]:
# 1. Average price by destination
price_by_dest = df_clean.groupby('destination_city')['price_cny'].agg(['mean', 'median', 'min', 'max']).round(0)
print("Price statistics by destination (CNY):")
print(price_by_dest)

# 2. Average price by holiday
price_by_holiday = df_clean.groupby('holiday')['price_cny'].mean().round(0)
print("\nAverage price by holiday:")
print(price_by_holiday)

# 3. Average price by holiday stage
price_by_stage = df_clean.groupby('holiday_stage')['price_cny'].mean().round(0)
print("\nAverage price by holiday stage:")
print(price_by_stage)

# 4. Cross-tab: destination × holiday stage
cross_price = df_clean.pivot_table(index='destination_city', columns='holiday_stage', values='price_cny', aggfunc='mean').round(0)
print("\nAverage price by destination and holiday stage (CNY):")
print(cross_price)

### Interpretation of Price Patterns

The charts show that prices are generally higher **during** the holiday period compared to before or after. Among destinations, Tokyo appears to have the highest average prices, while Seoul is relatively cheaper. The price increase during the holiday is most noticeable for Singapore and Tokyo.

These findings are useful for students and teachers because they suggest that traveling a few days before or after the official holiday period can lead to significant savings.

## 11. Comfort Analysis

This section examines how comfort-related features vary across routes and holiday periods.

The analysis focuses on:
- direct vs non-direct flights
- early-morning departures (red-eye)
- number of stops

In [ ]:
# 1. Compare price: direct vs non-direct
direct_price = df_clean.groupby('is_direct')['price_cny'].mean().round(0)
print("Average price by direct flight:")
print(direct_price)

# 2. Compare price: red-eye vs non-red-eye
redeye_price = df_clean.groupby('is_red_eye')['price_cny'].mean().round(0)
print("\nAverage price by red-eye flight:")
print(redeye_price)

# 3. Price by number of stops
stops_price = df_clean.groupby('num_stops')['price_cny'].mean().round(0)
print("\nAverage price by number of stops:")
print(stops_price)

# 4. Comfort indicators by destination
comfort_by_dest = df_clean.groupby('destination_city')[['is_direct_numeric', 'is_red_eye_numeric', 'num_stops']].mean()
print("\nComfort indicators by destination (proportion or average):")
print(comfort_by_dest.round(2))

### Interpretation of Comfort Patterns

The analysis shows that direct flights are on average more expensive than non-direct flights, and red-eye flights tend to be cheaper. As the number of stops increases, the average price decreases. This trade-off is important for students and teachers: the cheapest flight may involve an indirect route or an inconvenient departure time.

Across destinations, Tokyo has the highest proportion of direct flights, while Singapore has the lowest average number of stops.

## 12. Combining Price and Comfort

A good flight option is not necessarily the cheapest one. This section compares price and comfort together to identify options that may provide better overall value for students and teachers.

In [ ]:
## 10. Core Analysis: 4 Key Charts

fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# ========== Chart 1 ==========
sns.barplot(data=df_clean, x='destination_city', y='price_cny', errorbar=None, ax=axes[0,0])
axes[0,0].set_title('Chart 1: Average Price by Destination')
axes[0,0].set_xlabel('Destination')
axes[0,0].set_ylabel('Average Price (CNY)')

# --- Explanation for Chart 1 ---
axes[0,0].text(0.5, -0.25, 'Tokyo has the highest average price (1,540 CNY), while Seoul is the cheapest (1,267 CNY).\nThis matters for students because choosing Seoul over Tokyo can save about 270 CNY per trip.',
               transform=axes[0,0].transAxes, fontsize=9, ha='center')

# ========== Chart 2 ==========
sns.barplot(data=df_clean, x='holiday_stage', y='price_cny', errorbar=None, order=['before', 'during', 'after'], ax=axes[0,1])
axes[0,1].set_title('Chart 2: Average Price by Holiday Stage')
axes[0,1].set_xlabel('Holiday Stage')
axes[0,1].set_ylabel('Average Price (CNY)')

# --- Explanation for Chart 2 ---
axes[0,1].text(0.5, -0.25, 'Prices are highest BEFORE holidays (1,577 CNY) and drop significantly AFTER (840 CNY).\nStudents with flexible schedules can save nearly 50% by traveling after the holiday.',
               transform=axes[0,1].transAxes, fontsize=9, ha='center')

# ========== Chart 3 ==========
sns.barplot(data=df_clean, x='is_direct', y='price_cny', errorbar=None, ax=axes[1,0])
axes[1,0].set_title('Chart 3: Direct vs Non-direct Flight Price')
axes[1,0].set_xlabel('Is Direct')
axes[1,0].set_ylabel('Average Price (CNY)')

# --- Explanation for Chart 3 ---
axes[1,0].text(0.5, -0.25, 'Direct flights cost about 300 CNY more than non-direct flights.\nBudget-conscious students may consider indirect flights with 1 stop to save money.',
               transform=axes[1,0].transAxes, fontsize=9, ha='center')

# ========== Chart 4 ==========
sns.scatterplot(data=df_clean, x='comfort_score', y='price_cny', hue='destination_city', alpha=0.6, ax=axes[1,1])
axes[1,1].set_title('Chart 4: Price vs Comfort Score (Higher = More Comfortable)')
axes[1,1].set_xlabel('Comfort Score (0-3)')
axes[1,1].set_ylabel('Price (CNY)')
axes[1,1].legend(title='Destination')

# --- Explanation for Chart 4 ---
axes[1,1].text(0.5, -0.25, 'Higher comfort scores generally correlate with higher prices.\nSeoul offers the best balance: affordable flights with reasonable comfort levels.',
               transform=axes[1,1].transAxes, fontsize=9, ha='center')

plt.tight_layout()
plt.show()

# Print key statistics
print("\n=== Key Findings ===")
print(f"Cheapest destination: {df_clean.groupby('destination_city')['price_cny'].mean().idxmin()} "
      f"({df_clean.groupby('destination_city')['price_cny'].mean().min():.0f} CNY)")
print(f"Best time to buy: after holiday ({df_clean[df_clean['holiday_stage']=='after']['price_cny'].mean():.0f} CNY)")
print(f"Direct flights cost more by: {df_clean[df_clean['is_direct_numeric']==1]['price_cny'].mean() - df_clean[df_clean['is_direct_numeric']==0]['price_cny'].mean():.0f} CNY")

## 13. Key Findings

Based on the analysis, the main findings are:

1. **Price peaks before holidays** – The average ticket price before holidays (1,577 CNY) is much higher than after holidays (840 CNY), a difference of nearly 50%.

2. **Seoul is the most affordable destination** – Among the three destinations, Seoul has the lowest average price (1,267 CNY), while Tokyo is the most expensive (1,540 CNY).

3. **Direct flights cost a premium** – Direct flights are on average 300 CNY more expensive than indirect flights, which may be worth it for travelers who value time but not for budget-conscious students.

4. **Comfort comes at a cost** – Higher comfort scores (direct flights, non-red-eye, zero stops) are associated with higher prices. Seoul offers the best price-comfort balance.

## 14. Limitations

This project has several limitations:

- The dataset only covers selected routes (Shanghai to Singapore, Seoul, Tokyo) and two holidays (Labour Day, National Day).
- Ticket prices can change rapidly; the analysis reflects a snapshot in time (21 April 2026).
- Comfort is measured using only a few indicators (direct, red-eye, stops). Other factors like seat width, baggage allowance, or airline reputation are not included.
- The sample size for some categories (e.g., after-holiday period) is small.
- Actual crowd levels and airport congestion are not measured.

## 15. Conclusion

This project analyzed flight prices and comfort features for students and teachers traveling from Shanghai to Singapore, Seoul, and Tokyo during Labour Day and National Day holidays.

The results confirm that **during the holiday period**, prices are significantly higher than before or after. However, since students and teachers often cannot travel before or after the holiday due to school and work schedules, they must find the best available options within the holiday window itself.

Key findings within the holiday period:
- **Seoul** is the most affordable destination during holidays (1,331 CNY), while Tokyo is the most expensive (1,443 CNY)
- **Direct flights** cost a premium even during holidays — students on a tight budget may consider indirect options
- **Non-red-eye flights** are available at similar prices to red-eye flights for most routes, making them a better choice for comfort

For students and teachers who must travel during the holiday, the best strategy is:
- Choose **Seoul** over Tokyo or Singapore for better affordability
- Avoid direct flights if budget is the top priority
- Prioritize non-red-eye departures — they don't cost significantly more

Future work could compare price differences between booking early vs last-minute, or include more budget airlines that cater to student travelers.